In [10]:
import ROOT
import numpy as np

In [38]:
def fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime', 'weight': 'weight'}, SR='SR1'):
    f = ROOT.TFile(f'input/Run2/mc_signal_Hbb_{signal_mass}.root')
    tree = f.Get('Events')
    mass_Zprime = ROOT.RooRealVar(map['mass_Zprime'], "mass_Zprime", signal_mass, 650, 4000)
    weight = ROOT.RooRealVar(map['weight'], "weight", 0, -10, 10)
    mass_Higgs = ROOT.RooRealVar("mass_Higgs", "mass_Higgs", 125, 0, 999)
    tagger_Hbb = ROOT.RooRealVar("tagger_Hbb", "tagger_Hbb", 0, 0, 2)
    tagger_down, tagger_up = {'SR1': 0.7, 'SR2': 0.9}, {'SR1': 0.9, 'SR2': 2}
    SR_cut = f"(mass_Higgs>110) & (mass_Higgs<145) & (tagger_Hbb>{tagger_down[SR]}) & (tagger_Hbb<{tagger_up[SR]})"
    mc = ROOT.RooDataSet("signal_Hbb", "signal_Hbb", tree, ROOT.RooArgSet(mass_Zprime, weight, mass_Higgs, tagger_Hbb), SR_cut, map['weight'])
    
    x0 = ROOT.RooRealVar("x0", "x0", signal_mass, signal_mass - 200, signal_mass + 200)
    sigmaL = ROOT.RooRealVar("sigmaL", "sigmaL", 50, 1e-8, 200)
    sigmaR = ROOT.RooRealVar("sigmaR", "sigmaR", 50, 1e-8, 200)
    alphaL = ROOT.RooRealVar("alphaL", "alphaL", 2, 1e-8, 6)
    alphaR = ROOT.RooRealVar("alphaR", "alphaR", 2, 1e-8, 6)
    nL = ROOT.RooRealVar("nL", "nL", 1, 1e-8, 50)
    nR = ROOT.RooRealVar("nR", "nR", 1, 1e-8, 50)

    model_signal = ROOT.RooCrystalBall("model_signal", "model_signal", mass_Zprime, x0, sigmaL, sigmaR, alphaL, nL, alphaR, nR)
    model_signal.fitTo(mc, ROOT.RooFit.SumW2Error(True))
    model_signal.fitTo(mc, ROOT.RooFit.SumW2Error(True))

    """
    print("x0:", x0.getVal())
    print("sigmaL:", sigmaL.getVal())
    print("sigmaR:", sigmaR.getVal())
    print("alphaL:", alphaL.getVal())
    print("alphaR:", alphaR.getVal())
    print("nL:", nL.getVal())
    print("nR:", nR.getVal())
    """
    return x0.getVal(), sigmaL.getVal(), sigmaR.getVal()

In [39]:
parameters = {}

parameters['nominal'] = fit_signal(signal_mass=1000)
parameters['JES_up'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime_JES_up', 'weight': 'weight'})
parameters['JES_down'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime_JES_down', 'weight': 'weight'})
parameters['JER_up'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime_JER_up', 'weight': 'weight'})
parameters['JER_down'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime_JER_down', 'weight': 'weight'})
parameters['PU_up'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime', 'weight': 'weight_PU_up'})
parameters['PU_down'] = fit_signal(signal_mass=1000, map={'mass_Zprime': 'mass_Zprime', 'weight': 'weight_PU_down'})

In [ ]:
print('source\tmean\t\tsigmaL\t\tsigmaR')
for k in ['JES', 'JER', 'PU']:
    print(k, end='\t')
    print('%.4f/%.4f'%(parameters[f'{k}_down'][0]/parameters['nominal'][0], parameters[f'{k}_up'][0]/parameters['nominal'][0]), end='\t')
    print('%.3f/%.3f'%(parameters[f'{k}_down'][1]/parameters['nominal'][1], parameters[f'{k}_up'][1]/parameters['nominal'][1]), end='\t')
    print('%.3f/%.3f'%(parameters[f'{k}_down'][2]/parameters['nominal'][2], parameters[f'{k}_up'][2]/parameters['nominal'][2]))

source	mean		sigmaL		sigmaR
JES	0.9966/1.0035	1.013/0.993	0.997/1.002
JER	1.0010/0.9991	0.980/1.020	0.972/1.027
PU	1.0010/0.9993	0.992/1.014	0.995/1.001
